## 3. Selección de variables, modelado y evaluación

### 3.1 Control de Data Leakage

Columnas del dataset y su rol:

| Columna | Rol |
|---|---|
| `obra_id` | Identificador, se excluye |
| `superficie_m2`, `espesor_cm`, `densidad_suelo` | **Feature** |
| `tierra_teorica_kg` | Excluida (multicolinealidad con las 3 anteriores; no aporta mejora, ver 3.4) |
| `tipo_trabajo`, `tipo_suelo`, `estacion` | **Feature** (categóricas) |
| `pendiente_pct`, `nivelacion`, `accesibilidad`, `operarios`, `experiencia_operarios`, `lluvia_previa_mm` | **Feature** |
| `tierra_pedida_kg`, `tierra_sobrante_kg`, `tierra_consumida_real_kg` | **LEAKAGE - excluidas** (solo existen después de finalizar la obra) |
| `margen_necesario_pct` | **Target** |

### 3.2 Selección de variables: componentes vs `tierra_teorica_kg`

Se entrena con `superficie_m2 + espesor_cm + densidad_suelo` (sin `tierra_teorica_kg`), por dos motivos:
- Evita la multicolinealidad perfecta `tierra_teorica_kg = superficie_m2 * (espesor_cm/100) * densidad_suelo`.
- Es más natural para la demo (Streamlit): el usuario introduce superficie, espesor y densidad; `tierra_teorica_kg` se calcula internamente.

(Se probó también añadiendo `tierra_teorica_kg` como feature extra y no aportó mejora alguna, confirmando la redundancia.)

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import joblib

In [19]:
df = pd.read_csv('dataset_jardineria_tierra_limpio.csv')

In [20]:
feature_cols = ['superficie_m2', 'espesor_cm', 'densidad_suelo', 'tipo_trabajo',
                 'tipo_suelo', 'pendiente_pct', 'nivelacion', 'accesibilidad',
                 'operarios', 'experiencia_operarios', 'lluvia_previa_mm', 'estacion']
target_col = 'margen_necesario_pct'
 
X = df[feature_cols]
y = df[target_col]
 
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nNulos en X por columna:")
print(X.isna().sum()[X.isna().sum() > 0])

X shape: (656, 12)
y shape: (656,)

Nulos en X por columna:
superficie_m2            22
espesor_cm               23
densidad_suelo           23
tipo_suelo               25
pendiente_pct            26
nivelacion               19
accesibilidad            23
operarios                30
experiencia_operarios    23
lluvia_previa_mm         23
estacion                 25
dtype: int64


### 3.3 Split train/test

80/20, con `random_state` fijo para reproducibilidad. El split se hace ANTES de ajustar cualquier encoder, para evitar fuga de información entre train y test.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (524, 12) Test: (132, 12)


### 3.4 Preprocesado

One-Hot Encoding para `tipo_trabajo`, `tipo_suelo`, `estacion`. Las variables numéricas pasan sin transformar (`remainder='passthrough'`), conservando sus NaN.

Los NaN en categóricas se mantienen como una categoría propia ("nan"), permitiendo que el modelo aprenda de la ausencia de información.

Este `preprocessor` se integrará dentro de cada `Pipeline` de modelado: así, `fit()` ajusta el encoder SOLO con los datos de entrenamiento de cada fold/split, y `predict()` aplica siempre `transform()` — evitando leakage de forma automática.

In [21]:
cat_cols = ['tipo_trabajo', 'tipo_suelo', 'estacion']
num_cols = [c for c in feature_cols if c not in cat_cols]
 
print("Categóricas:", cat_cols)
print("Numéricas:", num_cols)
 
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ],
    remainder='passthrough'
)

Categóricas: ['tipo_trabajo', 'tipo_suelo', 'estacion']
Numéricas: ['superficie_m2', 'espesor_cm', 'densidad_suelo', 'pendiente_pct', 'nivelacion', 'accesibilidad', 'operarios', 'experiencia_operarios', 'lluvia_previa_mm']


### 3.5 Definición de modelos a comparar

- **Regresión Lineal** (baseline, requiere imputación de NaN)
- **XGBoost** (soporta NaN nativo)
- **LightGBM** (soporta NaN nativo)

Cada modelo se envuelve en un `Pipeline` junto con el `preprocessor` (y el `imputer` cuando es necesario), de modo que `fit`/`predict` ejecutan todo el flujo automáticamente.

In [23]:
models = {
    'Regresión Lineal': Pipeline([
        ('preprocessor', preprocessor),
        ('imputer', SimpleImputer(strategy='median')),
        ('model', LinearRegression())
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBRegressor(
            random_state=42, n_estimators=100, max_depth=3, learning_rate=0.05,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.8))
    ]),
    'LightGBM': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LGBMRegressor(
            random_state=42, n_estimators=100, max_depth=3, learning_rate=0.05,
            min_child_samples=15, subsample=0.8, colsample_bytree=0.8, verbose=-1))
    ]),
}

### 3.6 Validación cruzada (5-fold) sobre train

In [24]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
 
results = []
for name, pipe in models.items():
    mae = -cross_val_score(pipe, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error')
    rmse = -cross_val_score(pipe, X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')
    r2 = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2')
    results.append({
        'Modelo': name,
        'MAE': f"{mae.mean():.3f} (±{mae.std():.3f})",
        'RMSE': f"{rmse.mean():.3f} (±{rmse.std():.3f})",
        'R2': f"{r2.mean():.3f} (±{r2.std():.3f})",
    })
 
cv_results_df = pd.DataFrame(results)
print(cv_results_df.to_string(index=False))

c:\Users\adria\Desktop\TFM\tfm_materiales\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\adria\Desktop\TFM\tfm_materiales\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\adria\Desktop\TFM\tfm_materiales\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\adria\Desktop\TFM\tfm_materiales\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\adria\Desktop\TFM\tfm_materiales\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor wa

          Modelo            MAE           RMSE             R2
Regresión Lineal 1.791 (±0.142) 2.264 (±0.231) 0.613 (±0.078)
         XGBoost 1.860 (±0.115) 2.348 (±0.121) 0.587 (±0.033)
        LightGBM 1.870 (±0.108) 2.352 (±0.109) 0.586 (±0.028)


c:\Users\adria\Desktop\TFM\tfm_materiales\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\adria\Desktop\TFM\tfm_materiales\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\adria\Desktop\TFM\tfm_materiales\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


### 3.7 Interpretación de resultados de CV

| Modelo | MAE | RMSE | R2 |
|---|---|---|---|
| Regresión Lineal | 1.791 | 2.264 | 0.613 |
| XGBoost | 1.835 | 2.310 | 0.600 |
| LightGBM | 1.870 | 2.352 | 0.586 |

La **Regresión Lineal** obtiene el mejor resultado en las tres métricas, por delante de los modelos basados en árboles (XGBoost es el mejor de estos últimos, seguido de LightGBM).

Esto es coherente con el tamaño del dataset (524 obras en train) y el componente de ruido aleatorio introducido por diseño en la generación sintética (`N(0, 1.8)`): con pocos datos y ruido considerable, un modelo lineal generaliza mejor que modelos no lineales más complejos, que tienden a sobreajustar ligeramente.

**Se selecciona la Regresión Lineal como modelo final**, por ser la de mejor rendimiento, además de ser más simple, interpretable (coeficientes directos) y eficiente para el despliegue en la demo.

## 4. Entrenamiento final y evaluación en test

Se entrena el pipeline ganador con TODO el conjunto de train, y se evalúa en el conjunto de test (132 obras), que no se ha utilizado en ningún momento del proceso de selección de modelo.

In [25]:
final_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('imputer', SimpleImputer(strategy='median')),
    ('model', LinearRegression())
])
 
final_pipe.fit(X_train, y_train)
 
y_pred = final_pipe.predict(X_test)
mae_test = mean_absolute_error(y_test, y_pred)
rmse_test = mean_squared_error(y_test, y_pred) ** 0.5
r2_test = r2_score(y_test, y_pred)
 
print("Evaluación en TEST (132 obras):")
print(f"  MAE:  {mae_test:.3f}")
print(f"  RMSE: {rmse_test:.3f}")
print(f"  R2:   {r2_test:.3f}")

Evaluación en TEST (132 obras):
  MAE:  1.964
  RMSE: 2.476
  R2:   0.646


## 5. Interpretación: coeficientes del modelo

Una ventaja de la regresión lineal es la interpretabilidad directa de sus
coeficientes. Se comparan los signos con las reglas de negocio diseñadas:

- `nivelacion`, `accesibilidad`, `experiencia_operarios` -> coeficiente NEGATIVO esperado (mejoran estas variables -> menor margen necesario).
- `pendiente_pct`, `espesor_cm`, `lluvia_previa_mm` -> coeficiente POSITIVO esperado.
- `tipo_suelo_arcilloso` -> coeficiente POSITIVO esperado (mayor margen base); `tipo_suelo_arenoso` -> NEGATIVO esperado.

In [26]:
feature_names = final_pipe.named_steps['preprocessor'].get_feature_names_out()
coefs = final_pipe.named_steps['model'].coef_
intercept = final_pipe.named_steps['model'].intercept_
 
coef_df = pd.DataFrame({'feature': feature_names, 'coef': coefs})
coef_df = coef_df.sort_values('coef', key=abs, ascending=False)
 
print(f"Intercepto: {intercept:.4f}\n")
print(coef_df.to_string(index=False))

Intercepto: 11.3326

                         feature      coef
       cat__tipo_suelo_arcilloso  1.971045
         cat__tipo_suelo_arenoso -1.632209
           remainder__nivelacion -1.063401
           cat__tipo_suelo_mixto -0.772893
        remainder__accesibilidad -0.660421
   cat__tipo_trabajo_instalacion -0.581149
    cat__tipo_trabajo_renovacion  0.532981
             cat__tipo_suelo_nan  0.434056
           remainder__espesor_cm  0.327433
               cat__estacion_nan  0.321363
          cat__estacion_invierno -0.319921
remainder__experiencia_operarios -0.224126
        remainder__pendiente_pct  0.175659
            cat__estacion_verano  0.072742
             cat__estacion_otoño -0.063959
    cat__tipo_trabajo_ampliacion  0.048169
     remainder__lluvia_previa_mm  0.034533
            remainder__operarios -0.025116
         cat__estacion_primavera -0.010224
       remainder__densidad_suelo -0.001695
        remainder__superficie_m2  0.000177


Los signos de los coeficientes son coherentes con las reglas de negocio diseñadas: nivelación, accesibilidad y experiencia muestran coeficiente negativo; suelo arcilloso muestra el coeficiente positivo más alto y suelo arenoso uno de los más negativos; pendiente y espesor son positivos. Esto valida que el modelo ha aprendido relaciones consistentes con la lógica de negocio del proyecto.

## 6. Guardado del modelo para la demo (Streamlit)

In [27]:
joblib.dump(final_pipe, 'modelo_margen_tierra.pkl')
print("Modelo guardado: modelo_margen_tierra.pkl")

Modelo guardado: modelo_margen_tierra.pkl


El archivo `modelo_margen_tierra.pkl` contiene el pipeline completo (preprocesado + imputación + modelo), por lo que la app Streamlit puede cargarlo con `joblib.load()` y llamar directamente a `.predict()` pasando un DataFrame con las columnas originales (`feature_cols`), sin necesidad de repetir el preprocesado manualmente.